In [2]:
# -*- coding: utf-8 -*-
# =========================================================
# Hybrid Retrieval-Augmented Few-Shot
# ESG / Greenwashing Sentence Scoring with Local LLaMA
# Auto-save + Resume version
# 7 metrics version: + deflection + comparability
# =========================================================

# =========================================================
# 1. Imports
# =========================================================
import gc
import json
import os
import re
import time
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 2. Basic settings
# =========================================================
INPUT_FILE = "sentences_for_large_mark.csv"
OUTPUT_FILE = "greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark.csv"

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

SENTENCE_COL = "sentence"

MAX_NEW_TOKENS = 240
DO_SAMPLE = False
REPETITION_PENALTY = 1.02

USE_FEW_SHOT = True
USE_DYNAMIC_RETRIEVAL = True
RETRY_ON_FAIL = 1

# 固定 few-shot 保留幾個核心例子
N_CORE_FEW_SHOT = 2

# 動態檢索幾個 example
TOP_K_RETRIEVED = 2

# ===== 新增：自動存檔 / 續跑設定 =====
SAVE_EVERY = 1
CHECKPOINT_FILE = OUTPUT_FILE.replace(".csv", "_checkpoint.csv")
FALLBACK_FILE = OUTPUT_FILE.replace(".csv", "_fallback_rows.csv")
SUMMARY_FILE = OUTPUT_FILE.replace(".csv", "_summary.csv")

EXPECTED_KEYS = [
    "specificity",
    "evidence_substantiation",
    "vagueness",
    "commitment",
    "temporal_credibility",
    "deflection",
    "comparability"
]

SCORE_COLS = [
    "specificity_score",
    "evidence_substantiation_score",
    "vagueness_score",
    "commitment_score",
    "temporal_credibility_score",
    "deflection_score",
    "comparability_score"
]

FALLBACK_JSON = {
    "specificity": {"score": 0, "reason": "fallback"},
    "evidence_substantiation": {"score": 0, "reason": "fallback"},
    "vagueness": {"score": 3, "reason": "fallback"},
    "commitment": {"score": 0, "reason": "fallback"},
    "temporal_credibility": {"score": 0, "reason": "fallback"},
    "deflection": {"score": 0, "reason": "fallback"},
    "comparability": {"score": 0, "reason": "fallback"}
}

# =========================================================
# 3. Load data
# =========================================================
df = pd.read_csv(INPUT_FILE)
print("資料筆數:", len(df))

if SENTENCE_COL not in df.columns:
    raise ValueError(f"找不到欄位: {SENTENCE_COL}")

df[SENTENCE_COL] = df[SENTENCE_COL].astype(str).fillna("").str.strip()
df = df.reset_index(drop=True)
df["__row_id__"] = df.index

# 小測試可打開
# df = df.head(30).copy()

# =========================================================
# 4. Check GPU
# =========================================================
if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用，請確認你目前是在 GPU 環境。")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

# =========================================================
# 5. Clear memory
# =========================================================
gc.collect()
torch.cuda.empty_cache()

# =========================================================
# 6. Load LLaMA model
# =========================================================
print("Loading Llama model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_memory = {
    0: "7GiB",
    "cpu": "32GiB"
}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded successfully.")
print("Device map:", model.hf_device_map)

# =========================================================
# 7. Load embedding model
# =========================================================
print("Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL_ID)
print("Embedding model loaded:", EMBED_MODEL_ID)

# =========================================================
# 8. System prompt
# =========================================================
SYSTEM_PROMPT = """
You are an expert in sustainability communication and greenwashing analysis.

Task:
Evaluate the environmental sentence on exactly seven dimensions from 0 to 5:
1. specificity
2. evidence_substantiation
3. vagueness
4. commitment
5. temporal_credibility
6. deflection
7. comparability

Scoring reminders:
- specificity: higher = more measurable and concrete
- evidence_substantiation: higher = more proof, certification, audit, or supporting data
- vagueness: higher = more vague environmental wording
- commitment: higher = stronger future action commitment
- temporal_credibility: higher = clearer timeline, milestone, or deadline
- deflection: higher = stronger responsibility shifting to consumers, society, partners, or others
- comparability: higher = clearer baseline, benchmark, or comparison reference

STRICT OUTPUT RULES:
- Return ONLY one valid JSON object.
- Do NOT use markdown.
- Do NOT use code fences.
- Do NOT add any explanation before or after JSON.
- Use exactly these seven keys:
  specificity, evidence_substantiation, vagueness, commitment,
  temporal_credibility, deflection, comparability
- Each score must be an integer from 0 to 5.
- Each reason must be under 12 words.
- Start immediately with { and end with }.

Required format:
{
  "specificity":{"score":0,"reason":""},
  "evidence_substantiation":{"score":0,"reason":""},
  "vagueness":{"score":0,"reason":""},
  "commitment":{"score":0,"reason":""},
  "temporal_credibility":{"score":0,"reason":""},
  "deflection":{"score":0,"reason":""},
  "comparability":{"score":0,"reason":""}
}
""".strip()

# =========================================================
# 9. Core few-shot examples
# =========================================================
FEW_SHOT_EXAMPLES = [
    {
        "id": "core_1",
        "sentence": "We care about the planet and are working toward a greener future.",
        "answer": """{
"specificity":{"score":0,"reason":"No measurable details"},
"evidence_substantiation":{"score":0,"reason":"No evidence"},
"vagueness":{"score":5,"reason":"Highly vague sentence"},
"commitment":{"score":1,"reason":"Weak intention only"},
"temporal_credibility":{"score":0,"reason":"No timeline"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":0,"reason":"No baseline provided"}
}"""
    },
    {
        "id": "core_2",
        "sentence": "We aim to reduce water consumption by 25% across factories by 2028 compared with 2020 levels.",
        "answer": """{
"specificity":{"score":5,"reason":"Clear metric and scope"},
"evidence_substantiation":{"score":1,"reason":"No verification method"},
"vagueness":{"score":1,"reason":"Mostly concrete statement"},
"commitment":{"score":4,"reason":"Clear planned action"},
"temporal_credibility":{"score":4,"reason":"Defined deadline"},
"deflection":{"score":0,"reason":"Company keeps responsibility"},
"comparability":{"score":5,"reason":"Explicit baseline included"}
}"""
    },
    {
        "id": "core_3",
        "sentence": "Our packaging is more sustainable and better for the environment.",
        "answer": """{
"specificity":{"score":0,"reason":"No material details"},
"evidence_substantiation":{"score":0,"reason":"No supporting evidence"},
"vagueness":{"score":5,"reason":"Uses undefined terms"},
"commitment":{"score":1,"reason":"No concrete action"},
"temporal_credibility":{"score":0,"reason":"No timeframe stated"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":0,"reason":"No comparison baseline"}
}"""
    },
    {
        "id": "core_4",
        "sentence": "By 2030, all cotton in our products will be certified organic or recycled.",
        "answer": """{
"specificity":{"score":4,"reason":"Specific material deadline"},
"evidence_substantiation":{"score":1,"reason":"No proof mechanism"},
"vagueness":{"score":1,"reason":"Sentence is concrete"},
"commitment":{"score":5,"reason":"Strong future commitment"},
"temporal_credibility":{"score":5,"reason":"Clear end date"},
"deflection":{"score":0,"reason":"Company retains responsibility"},
"comparability":{"score":0,"reason":"No baseline comparison"}
}"""
    }
]

# =========================================================
# 10. Example bank for retrieval
# =========================================================
EXAMPLE_BANK = [
    {
        "id": "bank_1",
        "sentence": "Our operations are environmentally friendly and sustainable.",
        "answer": """{
"specificity":{"score":0,"reason":"No measurable information"},
"evidence_substantiation":{"score":0,"reason":"No supporting proof"},
"vagueness":{"score":5,"reason":"Highly vague wording"},
"commitment":{"score":1,"reason":"Weak environmental intention"},
"temporal_credibility":{"score":0,"reason":"No timeframe"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":0,"reason":"No baseline given"}
}"""
    },
    {
        "id": "bank_2",
        "sentence": "We will cut Scope 1 emissions by 40% by 2030.",
        "answer": """{
"specificity":{"score":5,"reason":"Clear metric and scope"},
"evidence_substantiation":{"score":1,"reason":"No verification specified"},
"vagueness":{"score":0,"reason":"Concrete statement"},
"commitment":{"score":5,"reason":"Strong commitment"},
"temporal_credibility":{"score":5,"reason":"Clear deadline"},
"deflection":{"score":0,"reason":"Company owns responsibility"},
"comparability":{"score":0,"reason":"No baseline stated"}
}"""
    },
    {
        "id": "bank_3",
        "sentence": "We support greener logistics and better packaging solutions.",
        "answer": """{
"specificity":{"score":1,"reason":"Very limited detail"},
"evidence_substantiation":{"score":0,"reason":"No evidence provided"},
"vagueness":{"score":4,"reason":"Broad positive wording"},
"commitment":{"score":2,"reason":"Weak intended action"},
"temporal_credibility":{"score":0,"reason":"No stated timeline"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":0,"reason":"No baseline reference"}
}"""
    },
    {
        "id": "bank_4",
        "sentence": "All manufacturing sites will use 100% renewable electricity by 2027.",
        "answer": """{
"specificity":{"score":5,"reason":"Specific target and scope"},
"evidence_substantiation":{"score":1,"reason":"No validation named"},
"vagueness":{"score":0,"reason":"Highly concrete sentence"},
"commitment":{"score":5,"reason":"Explicit future action"},
"temporal_credibility":{"score":5,"reason":"Clear target year"},
"deflection":{"score":0,"reason":"Company owns action"},
"comparability":{"score":0,"reason":"No comparison baseline"}
}"""
    },
    {
        "id": "bank_5",
        "sentence": "Our products are eco-friendly and designed for a better future.",
        "answer": """{
"specificity":{"score":0,"reason":"No concrete product detail"},
"evidence_substantiation":{"score":0,"reason":"No proof shown"},
"vagueness":{"score":5,"reason":"Undefined green terms"},
"commitment":{"score":1,"reason":"No strong commitment"},
"temporal_credibility":{"score":0,"reason":"No time reference"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":0,"reason":"No baseline given"}
}"""
    },
    {
        "id": "bank_6",
        "sentence": "We plan to reduce plastic packaging by 15% across Europe by 2026.",
        "answer": """{
"specificity":{"score":4,"reason":"Metric scope and action"},
"evidence_substantiation":{"score":1,"reason":"No substantiation given"},
"vagueness":{"score":1,"reason":"Mostly precise wording"},
"commitment":{"score":4,"reason":"Strong stated plan"},
"temporal_credibility":{"score":4,"reason":"Defined deadline"},
"deflection":{"score":0,"reason":"Company keeps responsibility"},
"comparability":{"score":0,"reason":"No baseline provided"}
}"""
    },
    {
        "id": "bank_7",
        "sentence": "Certified by FSC for all paper-based packaging materials.",
        "answer": """{
"specificity":{"score":3,"reason":"Some scope is defined"},
"evidence_substantiation":{"score":5,"reason":"Explicit certification stated"},
"vagueness":{"score":1,"reason":"Limited vague wording"},
"commitment":{"score":1,"reason":"Present status only"},
"temporal_credibility":{"score":0,"reason":"No timeline mentioned"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":0,"reason":"No baseline comparison"}
}"""
    },
    {
        "id": "bank_8",
        "sentence": "We aim to achieve net zero emissions in the long term.",
        "answer": """{
"specificity":{"score":1,"reason":"Target lacks detail"},
"evidence_substantiation":{"score":0,"reason":"No methodology included"},
"vagueness":{"score":4,"reason":"Long term is vague"},
"commitment":{"score":3,"reason":"Moderate intention stated"},
"temporal_credibility":{"score":1,"reason":"No clear deadline"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":0,"reason":"No baseline given"}
}"""
    },
    {
        "id": "bank_9",
        "sentence": "Third-party audited carbon data will be published annually starting in 2026.",
        "answer": """{
"specificity":{"score":4,"reason":"Clear reporting action"},
"evidence_substantiation":{"score":4,"reason":"Third-party audit mentioned"},
"vagueness":{"score":1,"reason":"Mostly precise wording"},
"commitment":{"score":4,"reason":"Clear future action"},
"temporal_credibility":{"score":4,"reason":"Start year specified"},
"deflection":{"score":0,"reason":"Company owns reporting"},
"comparability":{"score":0,"reason":"No comparison baseline"}
}"""
    },
    {
        "id": "bank_10",
        "sentence": "We are committed to making our business more sustainable.",
        "answer": """{
"specificity":{"score":0,"reason":"No measurable commitment"},
"evidence_substantiation":{"score":0,"reason":"No supporting basis"},
"vagueness":{"score":5,"reason":"Very broad wording"},
"commitment":{"score":2,"reason":"General commitment only"},
"temporal_credibility":{"score":0,"reason":"No timing given"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":0,"reason":"No baseline reference"}
}"""
    },
    {
        "id": "bank_11",
        "sentence": "Consumers can help reduce plastic waste by recycling our bottles.",
        "answer": """{
"specificity":{"score":2,"reason":"Some action mentioned"},
"evidence_substantiation":{"score":0,"reason":"No evidence given"},
"vagueness":{"score":3,"reason":"Limited operational detail"},
"commitment":{"score":1,"reason":"Weak company commitment"},
"temporal_credibility":{"score":0,"reason":"No timeline stated"},
"deflection":{"score":5,"reason":"Shifts responsibility consumers"},
"comparability":{"score":0,"reason":"No baseline provided"}
}"""
    },
    {
        "id": "bank_12",
        "sentence": "We reduced emissions by 25% compared with 2020 levels.",
        "answer": """{
"specificity":{"score":5,"reason":"Precise quantified reduction"},
"evidence_substantiation":{"score":1,"reason":"No verification cited"},
"vagueness":{"score":0,"reason":"Very concrete wording"},
"commitment":{"score":1,"reason":"Reports outcome only"},
"temporal_credibility":{"score":2,"reason":"Reference year provided"},
"deflection":{"score":0,"reason":"No responsibility shifting"},
"comparability":{"score":5,"reason":"Explicit baseline comparison"}
}"""
    }
]

# =========================================================
# 11. Prepare core few-shot examples
# =========================================================
CORE_FEW_SHOT_EXAMPLES = FEW_SHOT_EXAMPLES[:N_CORE_FEW_SHOT]
core_ids = set(ex["id"] for ex in CORE_FEW_SHOT_EXAMPLES)

# 避免 dynamic retrieval 把 core example 自己又抓進來
RETRIEVAL_BANK = [ex for ex in EXAMPLE_BANK if ex["id"] not in core_ids]

# =========================================================
# 12. Encode retrieval bank
# =========================================================
print("Encoding example retrieval bank...")

retrieval_texts = [x["sentence"] for x in RETRIEVAL_BANK]

retrieval_embeddings = embed_model.encode(
    retrieval_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Retrieval bank encoded:", len(RETRIEVAL_BANK), "examples")

# =========================================================
# 13. Helpers
# =========================================================
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def retrieve_similar_examples(sentence: str, top_k: int = 2):
    sentence = normalize_text(sentence)

    if len(RETRIEVAL_BANK) == 0:
        return []

    sentence_embedding = embed_model.encode(
        [sentence],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    sims = cosine_similarity(sentence_embedding, retrieval_embeddings)[0]
    top_indices = np.argsort(sims)[::-1][:top_k]

    retrieved = []
    for idx in top_indices:
        ex = RETRIEVAL_BANK[int(idx)].copy()
        ex["similarity"] = float(sims[idx])
        retrieved.append(ex)

    return retrieved


def save_checkpoint(results, checkpoint_file):
    checkpoint_df = pd.DataFrame(results)
    checkpoint_df.to_csv(checkpoint_file, index=False, encoding="utf-8-sig")


# =========================================================
# 14. Build messages
# =========================================================
def build_messages(sentence: str):
    sentence = normalize_text(sentence)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    retrieved_examples = []

    # 固定核心 few-shot
    if USE_FEW_SHOT:
        for ex in CORE_FEW_SHOT_EXAMPLES:
            messages.append({"role": "user", "content": f"Sentence: {ex['sentence']}"})
            messages.append({"role": "assistant", "content": ex["answer"]})

    # 動態 retrieval examples
    if USE_DYNAMIC_RETRIEVAL:
        retrieved_examples = retrieve_similar_examples(sentence, top_k=TOP_K_RETRIEVED)

        for ex in retrieved_examples:
            messages.append({"role": "user", "content": f"Sentence: {ex['sentence']}"})
            messages.append({"role": "assistant", "content": ex["answer"]})

    # target sentence
    messages.append({"role": "user", "content": f"Sentence: {sentence}"})

    return messages, retrieved_examples


def build_repair_messages(sentence: str):
    sentence = normalize_text(sentence)

    repair_system_prompt = """
You are a strict JSON generator.

Evaluate the environmental sentence on exactly seven dimensions from 0 to 5:
1. specificity
2. evidence_substantiation
3. vagueness
4. commitment
5. temporal_credibility
6. deflection
7. comparability

Scoring reminders:
- specificity: higher = more measurable and concrete
- evidence_substantiation: higher = more proof, certification, audit, or supporting data
- vagueness: higher = more vague environmental wording
- commitment: higher = stronger future action commitment
- temporal_credibility: higher = clearer timeline, milestone, or deadline
- deflection: higher = stronger responsibility shifting to others
- comparability: higher = clearer baseline or comparison

Return ONLY one valid JSON object.
Do not add any extra text.
Do not use markdown.
Do not use code fences.
Scores must be integers from 0 to 5.
Reasons must be very short, under 8 words.

Required format:
{
"specificity":{"score":0,"reason":""},
"evidence_substantiation":{"score":0,"reason":""},
"vagueness":{"score":0,"reason":""},
"commitment":{"score":0,"reason":""},
"temporal_credibility":{"score":0,"reason":""},
"deflection":{"score":0,"reason":""},
"comparability":{"score":0,"reason":""}
}
""".strip()

    return [
        {"role": "system", "content": repair_system_prompt},
        {"role": "user", "content": f"Sentence: {sentence}"}
    ]

# =========================================================
# 15. JSON helpers
# =========================================================
def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    text = text.replace("```json", "").replace("```", "").strip()
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("\u00a0", " ")
    return text


def extract_first_json_block(text: str):
    """
    用大括號平衡法抓第一個完整 JSON object
    """
    text = clean_text(text)
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def repair_json_string(s: str) -> str:
    if s is None:
        return None

    s = s.strip()

    s = re.sub(r",\s*}", "}", s)
    s = re.sub(r",\s*]", "]", s)
    s = s.replace("\r", "")

    if '"' not in s and "'" in s:
        s = s.replace("'", '"')

    open_braces = s.count("{")
    close_braces = s.count("}")

    if close_braces < open_braces:
        s = s + ("}" * (open_braces - close_braces))

    return s


def normalize_parsed_json(obj):
    if not isinstance(obj, dict):
        return obj

    for key in EXPECTED_KEYS:
        if key in obj and isinstance(obj[key], dict):
            score = obj[key].get("score", None)
            reason = obj[key].get("reason", "")

            if isinstance(score, str):
                score_str = score.strip()
                if score_str.isdigit():
                    obj[key]["score"] = int(score_str)

            if reason is None:
                obj[key]["reason"] = ""
            else:
                obj[key]["reason"] = str(reason)

    return obj


def validate_parsed_json(obj):
    if not isinstance(obj, dict):
        return False

    for key in EXPECTED_KEYS:
        if key not in obj:
            return False

        val = obj[key]
        if not isinstance(val, dict):
            return False

        if "score" not in val or "reason" not in val:
            return False

        score = val["score"]
        reason = val["reason"]

        if not isinstance(score, int):
            return False
        if score < 0 or score > 5:
            return False
        if not isinstance(reason, str):
            return False

    return True


def extract_json(text: str):
    cleaned = clean_text(text)

    json_str = extract_first_json_block(cleaned)

    if json_str is None:
        start = cleaned.find("{")
        if start == -1:
            return None
        json_str = cleaned[start:]

    json_str = repair_json_string(json_str)

    try:
        parsed = json.loads(json_str)
    except Exception:
        return None

    parsed = normalize_parsed_json(parsed)

    if not validate_parsed_json(parsed):
        return None

    return parsed

# =========================================================
# 16. Generation
# =========================================================
def generate(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text

# =========================================================
# 17. Main scoring loop (auto-save + resume)
# =========================================================
results = []
start_time = time.time()
processed_indices = set()

if os.path.exists(CHECKPOINT_FILE):
    checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
    results = checkpoint_df.to_dict(orient="records")

    if "__row_id__" in checkpoint_df.columns:
        processed_indices = set(checkpoint_df["__row_id__"].tolist())

    print(f"讀到 checkpoint，共 {len(results)} 筆，將續跑剩餘資料。")
else:
    print("沒有 checkpoint，從頭開始跑。")

for i, row in tqdm(df.iterrows(), total=len(df)):
    row_id = row["__row_id__"]

    if row_id in processed_indices:
        continue

    sentence = str(row[SENTENCE_COL]).strip()
    raw = ""
    parsed = None
    retrieved_examples = []

    try:
        # ---------- first pass ----------
        messages, retrieved_examples = build_messages(sentence)
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        raw = generate(prompt)
        parsed = extract_json(raw)

        # ---------- retry same prompt ----------
        retry_count = 0
        while parsed is None and retry_count < RETRY_ON_FAIL:
            raw = generate(prompt)
            parsed = extract_json(raw)
            retry_count += 1

        # ---------- repair prompt ----------
        if parsed is None:
            repair_messages = build_repair_messages(sentence)
            repair_prompt = tokenizer.apply_chat_template(
                repair_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            raw = generate(repair_prompt)
            parsed = extract_json(raw)

    except Exception as e:
        raw = f"ERROR: {str(e)}"
        parsed = None
        retrieved_examples = []

    print(f"RAW {i+1}:", raw[:180])

    row_dict = row.to_dict()
    row_dict["raw_llm_output"] = raw
    row_dict["prompt_type"] = "hybrid_retrieval_augmented_fewshot_7metrics"
    row_dict["core_example_ids"] = "|".join([x["id"] for x in CORE_FEW_SHOT_EXAMPLES])

    if len(retrieved_examples) > 0:
        row_dict["retrieved_example_ids"] = "|".join([x["id"] for x in retrieved_examples])
        row_dict["retrieved_example_sentences"] = " ||| ".join([x["sentence"] for x in retrieved_examples])
        row_dict["retrieved_example_similarities"] = "|".join([f"{x['similarity']:.4f}" for x in retrieved_examples])
    else:
        row_dict["retrieved_example_ids"] = ""
        row_dict["retrieved_example_sentences"] = ""
        row_dict["retrieved_example_similarities"] = ""

    if parsed is None:
        parsed = FALLBACK_JSON
        row_dict["json_parse_success"] = False
        row_dict["used_fallback"] = True
    else:
        row_dict["json_parse_success"] = True
        row_dict["used_fallback"] = False

    for key in EXPECTED_KEYS:
        row_dict[f"{key}_score"] = parsed[key]["score"]
        row_dict[f"{key}_reason"] = parsed[key]["reason"]

    results.append(row_dict)
    processed_indices.add(row_id)

    if len(results) % SAVE_EVERY == 0:
        save_checkpoint(results, CHECKPOINT_FILE)
        print(f"[checkpoint] 已儲存 {len(results)} 筆 -> {CHECKPOINT_FILE}")

elapsed = time.time() - start_time

# =========================================================
# 18. Save final results
# =========================================================
df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print("Saved final:", OUTPUT_FILE)

# =========================================================
# 19. Basic stats
# =========================================================
print("\nJSON success:")
print(df_out["json_parse_success"].value_counts(dropna=False))

print("\nUsed fallback:")
print(df_out["used_fallback"].value_counts(dropna=False))

print("\nMissing values:")
print(df_out[SCORE_COLS].isna().sum())

if "label" in df_out.columns:
    print("\nLabel mean:")
    grp = df_out.groupby("label")[SCORE_COLS].mean()
    print(grp)

    if 0 in grp.index and 1 in grp.index:
        gap = grp.loc[0] - grp.loc[1]
        print("\nLabel gap (label 0 - label 1):")
        print(gap)

        print("\nDirection check (expected signs):")
        expected_sign = {
            "specificity_score": "+",
            "evidence_substantiation_score": "+",
            "vagueness_score": "-",
            "commitment_score": "+",
            "temporal_credibility_score": "+",
            "deflection_score": "-",
            "comparability_score": "+"
        }

        for col in SCORE_COLS:
            val = gap[col]
            actual = "+" if val > 0 else "-" if val < 0 else "0"
            print(f"{col}: gap={val:.6f}, expected={expected_sign[col]}, actual={actual}")

# =========================================================
# 20. Save fallback rows separately
# =========================================================
fallback_df = df_out[df_out["used_fallback"] == True].copy()
if len(fallback_df) > 0:
    fallback_df.to_csv(FALLBACK_FILE, index=False, encoding="utf-8-sig")
    print("\nFallback rows saved:", FALLBACK_FILE)

# =========================================================
# 21. Save retrieval summary
# =========================================================
summary_rows = []

summary_rows.append({"metric": "n_rows", "value": len(df_out)})
summary_rows.append({"metric": "elapsed_seconds", "value": round(elapsed, 2)})
summary_rows.append({"metric": "elapsed_minutes", "value": round(elapsed / 60, 2)})
summary_rows.append({"metric": "json_success_rate", "value": round(df_out["json_parse_success"].mean(), 6)})
summary_rows.append({"metric": "fallback_rate", "value": round(df_out["used_fallback"].mean(), 6)})
summary_rows.append({"metric": "n_core_fewshot", "value": N_CORE_FEW_SHOT})
summary_rows.append({"metric": "n_dynamic_retrieved", "value": TOP_K_RETRIEVED})
summary_rows.append({"metric": "retrieval_bank_size", "value": len(RETRIEVAL_BANK)})

if "label" in df_out.columns:
    grp = df_out.groupby("label")[SCORE_COLS].mean()
    if 0 in grp.index and 1 in grp.index:
        gap = grp.loc[0] - grp.loc[1]
        for col in SCORE_COLS:
            summary_rows.append({
                "metric": f"label_gap_{col}",
                "value": round(float(gap[col]), 6)
            })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_FILE, index=False, encoding="utf-8-sig")
print("Summary saved:", SUMMARY_FILE)

print(f"\nTotal time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
print("\nDone.")

資料筆數: 472
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.59 GB
Loading Llama model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.00s/it]


Model loaded successfully.
Device map: {'': 0}
Loading embedding model...
Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Encoding example retrieval bank...
Retrieval bank encoded: 12 examples
讀到 checkpoint，共 21 筆，將續跑剩餘資料。


  0%|          | 0/472 [00:00<?, ?it/s]c:\greenwashingllm\llama_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\greenwashingllm\llama_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
  5%|▍         | 22/472 [00:10<03:31,  2.13it/s]

RAW 22: {
"specificity":{"score":5,"reason":"Concrete metric and scope"},
"evidence_substantiation":{"score":3,"reason":"Supporting data provided"},
"vagueness":{"score":1,"reason":"Mostly
[checkpoint] 已儲存 22 筆 -> greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark_checkpoint.csv


...

RAW 470: {
"specificity":{"score":4,"reason":"Clear goal and process mentioned"},
"evidence_substantiation":{"score":2,"reason":"Process described, but no results"},
"vagueness":{"score":2,
[checkpoint] 已儲存 470 筆 -> greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark_checkpoint.csv


c:\greenwashingllm\llama_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\greenwashingllm\llama_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
100%|█████████▉| 471/472 [1:17:47<00:10, 10.37s/it]

RAW 471: {
"specificity":{"score":5,"reason":"Detailed metric and scope"},
"evidence_substantiation":{"score":4,"reason":"Named company and year verified"},
"vagueness":{"score":0,"reason":
[checkpoint] 已儲存 471 筆 -> greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark_checkpoint.csv


c:\greenwashingllm\llama_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\greenwashingllm\llama_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
100%|██████████| 472/472 [1:17:58<00:00,  9.91s/it]

RAW 472: {
"specificity":{"score":5,"reason":"Multiple specific metrics mentioned"},
"evidence_substantiation":{"score":4,"reason":"CDP survey results mentioned"},
"vagueness":{"score":0,"r
[checkpoint] 已儲存 472 筆 -> greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark_checkpoint.csv
Saved final: greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark.csv

JSON success:
json_parse_success
True    472
Name: count, dtype: int64

Used fallback:
used_fallback
False    472
Name: count, dtype: int64

Missing values:
specificity_score                0
evidence_substantiation_score    0
vagueness_score                  0
commitment_score                 0
temporal_credibility_score       0
deflection_score                 0
comparability_score              0
dtype: int64
Summary saved: greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark_summary.csv

Total time: 4678.15 

In [3]:
df = pd.read_csv("greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark.csv")

print("=== 基本 ===")
print("筆數:", len(df))
print("unique row_id:", df["__row_id__"].nunique())

print("\n=== JSON ===")
print(df["json_parse_success"].value_counts())
print("fallback rate:", df["used_fallback"].mean())

print("\n=== NA ===")
print(df.isna().sum())

print("\n=== 分布 ===")
print(df["specificity_score"].describe())
print(df["vagueness_score"].describe())

=== 基本 ===
筆數: 472
unique row_id: 472

=== JSON ===
json_parse_success
True    472
Name: count, dtype: int64
fallback rate: 0.0

=== NA ===
sentence_id                        0
folder_year                        0
sic                                0
company                            0
ticker                            56
report_year                        0
source_folder                      0
file                               0
rank                               0
score                              0
page                               0
section_guess                      0
topic_guess                        0
sentence_type                      0
has_number                         0
has_year                           0
has_by_year                        0
has_percent                        0
has_scope                          0
has_sbti                           0
has_netzero                        0
has_kpi                            0
has_material                       0
has_green

In [4]:
metrics = [
    "specificity_score",
    "evidence_substantiation_score",
    "vagueness_score",
    "commitment_score",
    "temporal_credibility_score",
    "deflection_score",
    "comparability_score"
]

df[metrics].describe()

,specificity_score,evidence_substantiation_score,vagueness_score,commitment_score,temporal_credibility_score,deflection_score,comparability_score
count,472.000000,472.000000,472.000000,472.000000,472.000000,472.000000,472.000000
mean,4.680085,2.766949,0.516949,4.241525,4.046610,0.230932,3.779661
std,0.746816,1.201878,0.874096,0.988754,1.164746,0.745586,1.118352
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.000000,2.000000,0.000000,4.000000,4.000000,0.000000,3.000000
50%,5.000000,3.000000,0.000000,4.000000,4.000000,0.000000,4.000000
75%,5.000000,4.000000,1.000000,5.000000,5.000000,0.000000,4.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,4.000000,5.000000
